# PRE-PROCESSING Face Spoof Detector DATASET

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [3]:
import cv2
import mediapipe as mp
import os
from pathlib import Path
from tqdm import tqdm # Progress bar
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- File paths ---
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite"
MODEL_PATH = "face_detector.tflite"
DRIVE_BASE = "/content/drive/MyDrive/Deep_Learning_CV_projects/final_project"
LOCAL_DATASET = "/content/dataset"



In [4]:
# Download model: https://ai.google.dev/edge/mediapipe/solutions/vision/face_detector#blazeface_short-range
"""
BlazeFace Short Range (from google AI for devs)

A lightweight model for detecting single or multiple faces within selfie-like images from a smartphone camera or webcam.
The model is optimized for front-facing phone camera images at short range.
The model architecture uses a Single Shot Detector (SSD) convolutional network technique with a custom encoder.
"""

!wget -O face_detector.tflite {MODEL_URL}

--2026-04-24 22:45:22--  https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.98.207, 142.251.107.207, 74.125.196.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.98.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 229746 (224K) [application/octet-stream]
Saving to: ‘face_detector.tflite’

face_detector.tflit 100%[===================>] 224.36K  --.-KB/s    in 0.002s  

2026-04-24 22:45:22 (130 MB/s) - ‘face_detector.tflite’ saved [229746/229746]



In [5]:
# 1. Initialize the MediaPipe Task Detector
def get_detector():

    # create object base options to configure Mediapipe task
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

    # create options object to configure Face detector
    options = vision.FaceDetectorOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE, # run on images, not the full video
        min_detection_confidence=0.5 # if less, detected face will not be considered valid
    )
    return vision.FaceDetector.create_from_options(options)



In [6]:

# 2. Processing Engine
def extract_face_crops(input_folder, output_folder, label, frame_rate=5):
    detector = get_detector()
    # make sure output_folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Identify all .mp4 and .mov files (case-insensitive)
    video_files = [f for f in Path(input_folder).iterdir() if f.suffix.lower() in ['.mp4', '.mov']]

    print(f"\n--- Processing {len(video_files)} videos for: {label} ---")

    for video_path in tqdm(video_files, desc=f"Extracting {label}"):
        # Create video capture object
        cap = cv2.VideoCapture(str(video_path))
        # Get certain property from video, in this case FPS(frame rate) this should be 30 from my IPhone
        fps = cap.get(cv2.CAP_PROP_FPS)

        if fps == 0: continue # Skip if video can't be read

        interval = max(1, int(fps / frame_rate)) # get interval we want to extract
        frame_idx = 0
        saved_count = 0

        while cap.isOpened():
            # capture frame by frame , returns true if succesful
            ret, frame = cap.read()
            if not ret: break

            if frame_idx % interval == 0:
                # Convert BGR to RGB for MediaPipe
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

                # PASS TO DETECTOR
                result = detector.detect(mp_image)

                if result.detections:
                    # Capture the primary face detection( Red Bounding box)
                    bbox = result.detections[0].bounding_box
                    h, w, _ = frame.shape

                    # Coordinates from MediaPipe Tasks
                    x, y, bw, bh = bbox.origin_x, bbox.origin_y, bbox.width, bbox.height

                    # Apply 20% Margin (Captures screen texture and surrounding context)
                    margin_w, margin_h = int(bw * 0.2), int(bh * 0.2)
                    x1, y1 = max(0, x - margin_w), max(0, y - margin_h)
                    x2, y2 = min(w, x + bw + margin_w), min(h, y + bh + margin_h)

                    # Crop and Normalize to 224x224
                    crop = frame[y1:y2, x1:x2] # array slicing to extract rectangular region
                    if crop.size > 0:
                        crop = cv2.resize(crop, (224, 224))
                        filename = f"{label}_{video_path.stem}_f{frame_idx}.jpg"
                        cv2.imwrite(os.path.join(output_folder, filename), crop)
                        saved_count += 1

            frame_idx += 1
        cap.release()


In [7]:
# 3. EXECUTION


extract_face_crops(f"{DRIVE_BASE}/Real_Videos", f"{LOCAL_DATASET}/all/real", "real")
extract_face_crops(f"{DRIVE_BASE}/Spoof_Videos", f"{LOCAL_DATASET}/all/spoof", "spoof")


--- Processing 10 videos for: real ---


Extracting real: 100%|██████████| 10/10 [03:36<00:00, 21.69s/it]



--- Processing 10 videos for: spoof ---


Extracting spoof: 100%|██████████| 10/10 [03:08<00:00, 18.83s/it]


# Save to drive

In [8]:
import os

# --- PATHS ---
# Colab Local sources from the extraction step
LOCAL_REAL = "/content/dataset/all/real"
LOCAL_SPOOF = "/content/dataset/all/spoof"

# Drive destination folders
DRIVE_DEST_REAL = "/content/drive/MyDrive/Deep_Learning_CV_projects/final_project/Real_Images"
DRIVE_DEST_SPOOF = "/content/drive/MyDrive/Deep_Learning_CV_projects/final_project/Spoof_Images"

def backup_to_persistent_storage():
    sync_tasks = [
        (LOCAL_REAL, DRIVE_DEST_REAL),
        (LOCAL_SPOOF, DRIVE_DEST_SPOOF)
    ]

    for src, dest in sync_tasks:
        if not os.path.exists(src):
            print(f"Source folder {src} not found. Ensure extraction finished.")
            continue

        print(f"Backing up: {src} -> {dest}")
        os.makedirs(dest, exist_ok=True)

        # -r: recursive copy
        # -n: do not overwrite files that already exist in the destination
        # Using shell command directly for high-speed transfer of many small files
        !cp -rn {src}/. {dest}/

        # Verification count
        final_count = len(os.listdir(dest))
        print(f"Success. Current total in Drive folder: {final_count} images.")

backup_to_persistent_storage()

Backing up: /content/dataset/all/real -> /content/drive/MyDrive/Deep_Learning_CV_projects/final_project/Real_Images
Success. Current total in Drive folder: 1286 images.
Backing up: /content/dataset/all/spoof -> /content/drive/MyDrive/Deep_Learning_CV_projects/final_project/Spoof_Images
Success. Current total in Drive folder: 1293 images.
